In [1]:
# Installa el cliente de InfluxDB, polars y python-dotenv para Python junto a paquetes necesarios
%pip install influxdb-client
%pip install polars
%pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importar polars como pl y
import polars as pl

In [3]:
# Importa la librería de InfluxDB para Python
import influxdb_client

In [4]:
# Crea un cliente de InfluxDB con la URL, token y organización especificados en el entorno (.env)
import os
from dotenv import load_dotenv
from influxdb_client import InfluxDBClient

# Cargar configuración desde PIA/.env si existe, o desde variables del sistema
load_dotenv(dotenv_path="../PIA/.env")

INFLUX_URL = os.getenv("INFLUXDB_URL", "http://localhost:8086")
INFLUX_TOKEN = os.getenv("DOCKER_INFLUXDB_INIT_ADMIN_TOKEN", "tokensomorrostro123456789")
INFLUX_ORG = os.getenv("DOCKER_INFLUXDB_INIT_ORG", "orgsomorrostro")

client = InfluxDBClient(
    url=INFLUX_URL,
    token=INFLUX_TOKEN,
    org=INFLUX_ORG
)

print(client.ping())

True


In [5]:
# Lee el archivo CSV "steam_data.csv" y lo carga en un DataFrame de Polars
df = pl.read_csv("steam_data.csv")

In [6]:
df = pl.read_csv(
    "steam_data.csv",
    separator=";",
    try_parse_dates=True
)

df_juegos = df.filter(
    pl.col("Game").str.contains("(?i)counter.?strike|among us")
)

print(df_juegos.select("Game").unique())
print(df_juegos.shape)
print(df_juegos.head(2))

shape: (2, 1)
┌──────────────────┐
│ Game             │
│ ---              │
│ str              │
╞══════════════════╡
│ Counter-Strike 2 │
│ Among Us         │
└──────────────────┘
(11384, 4)
shape: (2, 4)
┌──────────┬─────────────────────┬─────────┬─────────────────┐
│ Game     ┆ DateTime            ┆ Players ┆ Average Players │
│ ---      ┆ ---                 ┆ ---     ┆ ---             │
│ str      ┆ datetime[μs]        ┆ i64     ┆ str             │
╞══════════╪═════════════════════╪═════════╪═════════════════╡
│ Among Us ┆ 2018-11-16 00:00:00 ┆ 8       ┆                 │
│ Among Us ┆ 2018-11-17 00:00:00 ┆ 17      ┆                 │
└──────────┴─────────────────────┴─────────┴─────────────────┘


In [7]:
# Convierte la columna "Average Players" a tipo float64, eliminando las comas y reemplazándolas por nada
df_envio = df_juegos.with_columns(
    pl.col("Average Players")
      .str.replace_all(",", "")
      .cast(pl.Float64, strict=False)
)

In [ ]:
# Subir los datos a InfluxDB
import os
import time
from dotenv import load_dotenv
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

load_dotenv(dotenv_path="../PIA/.env")

INFLUX_URL = os.getenv("INFLUXDB_URL", "http://localhost:8086")
INFLUX_TOKEN = os.getenv("DOCKER_INFLUXDB_INIT_ADMIN_TOKEN", "tokensomorrostro123456789")
INFLUX_ORG = os.getenv("DOCKER_INFLUXDB_INIT_ORG", "orgsomorrostro")

client = InfluxDBClient(
    url=INFLUX_URL,
    token=INFLUX_TOKEN,
    org=INFLUX_ORG,
    timeout=60000
)

# Crear el bucket si no existe
bucket = "Copia seguridad"
buckets_api = client.buckets_api()
if buckets_api.find_bucket_by_name(bucket) is None:
    buckets_api.create_bucket(bucket_name=bucket, org=INFLUX_ORG)
    print(f"Bucket '{bucket}' creado con éxito.")
else:
    print(f"El bucket '{bucket}' ya existe.")

write_api = client.write_api(write_options=SYNCHRONOUS)

# Crear los puntos a partir del DataFrame filtrado de Polars (ordenado cronológicamente)
puntos = []

for fila in df_envio.iter_rows(named=True):
    punto = (
        Point("steam_data")
        .tag("game", fila["Game"])
        .time(fila["DateTime"], WritePrecision.US)
    )

    if fila["Players"] is not None:
        punto = punto.field("players", int(fila["Players"]))

    if fila["Average Players"] is not None:
        punto = punto.field("average_players", float(fila["Average Players"]))

    puntos.append(punto)

# Subirlos en lotes con reintentos para evitar sobrecargar los shards del motor de InfluxDB
lote_tamano = 200

for inicio in range(0, len(puntos), lote_tamano):
    lote = puntos[inicio:inicio + lote_tamano]
    for intento in range(3):
        try:
            write_api.write(bucket=bucket, org=INFLUX_ORG, record=lote)
            break
        except Exception:
            time.sleep(1)
    if (inicio + lote_tamano) % 1000 < lote_tamano or (inicio + lote_tamano) >= len(puntos):
        print(f"Enviados {min(inicio + lote_tamano, len(puntos))} de {len(puntos)}")

client.close()

Enviados 500 de 11384
Enviados 1000 de 11384
Enviados 1500 de 11384
Enviados 2000 de 11384
Enviados 2500 de 11384
Enviados 3000 de 11384
Enviados 3500 de 11384
Enviados 4000 de 11384
Enviados 4500 de 11384
Enviados 5000 de 11384
Enviados 5500 de 11384
Enviados 6000 de 11384
Enviados 6500 de 11384
Enviados 7000 de 11384
Enviados 7500 de 11384
Enviados 8000 de 11384
Enviados 8500 de 11384
Enviados 9000 de 11384
Enviados 9500 de 11384
Enviados 10000 de 11384
Enviados 10500 de 11384
Enviados 11000 de 11384
Enviados 11384 de 11384
